# Lab 5: Random Forest Practice

Apply a Random Forest classifier, `RandomForest()` to the heart disease dataset

The data is hosted on UCI https://archive.ics.uci.edu/ml/datasets/Heart+Disease

We will need `load_heart_disease()` from the `Linear Example.ipynb` file from Lab 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Load and prepare the data
**Question:** Can you update the function `load_heart_disease()` to download the file from the url if the file is not on disk?

In [ ]:
def load_heart_disease():
    '''Load and pre-process heart disease data
    
    if processed.hungarian.data file is not present.
    
    it will be downloaded from
    https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.hungarian.data
    
    return: data(DataFrame)
    
    '''
    
    import os
    import requests
    
    
    file_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.hungarian.data'
    file_name = file_url.split('/')[-1]
    
    if not os.path.isfile(file_name):
        print('Downloading from {}'.format(file_url))
        r = requests.get(file_url)
        with open(file_name,'wb') as output_file:
            output_file.write(r.content)
    
    # Read in data
    data = pd.read_csv(file_name, 
                   na_values='?', 
                   names=[ 'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
                            'restecg', 'thalach', 'exang', 'oldpeak', 'slope',
                            'ca', 'thal', 'num'])
    
    # drop columns with many missing data
    data = data.drop(columns=['slope', 'ca', 'thal'])
    
    # fill in remaining missing data with mean() per column
    data = data.fillna(data.mean())
    
    return data

In [ ]:
# Load data
data = load_heart_disease()
data.info()

### Create feature matrix and target vector 

In [ ]:
# Create feature matrix and target vector and print shapes
X = data.drop(columns='num')
y = data['num']
print(X.shape)
print(y.shape)

### Create training and test sets

In [ ]:
# Leave 10% of the data for testing. Use random_state=31
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.1,
                                                    stratify=y,
                                                    random_state=31)

print(X_train.shape)
print(X_test.shape)

In [ ]:
y_train.value_counts()

In [ ]:
y_test.value_counts()

## Apply `RandomForest()` and evaluate the performance 

In [ ]:
# Create and train model object
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=0)
model.fit(X_train, y_train)

### Cross validation 

Are we over- or underfitting?

Using `cross_val_score()`

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X_train, y_train, cv=5)
scores 

In [ ]:
#Calculate mean
scores.mean()

Using `cross_validate()`

In [ ]:
from sklearn.model_selection import cross_validate
scores = cross_validate(model, X_train, y_train, cv=5, 
                        scoring='accuracy',
                       return_train_score=True)
scores

In [ ]:
# Calculate means
for label_pair in [ ('train_score', 'train_score'), ('test_score', 'validation_score')]:
    print('{} = {:.3f}'.format(label_pair[1], scores[label_pair[0]].mean()))

Likely overfitting and would need a less complex model to increase validation accuracy.

We could use: Pre-pruning by reducing maximum depth. More trees, or smaller max_features to reduce overfitting.

### Compare training and test accuracy


In [ ]:
# Training accuracy
model.score(X_train, y_train)

In [ ]:
# Testing accuracy
model.score(X_test, y_test)

### The confusion matrix on the test set 

Your thoughts?

In [ ]:
# Display confusion matrix
from sklearn.metrics import confusion_matrix

mat = confusion_matrix(y_test, model.predict(X_test))

sns.heatmap(mat, square=True, annot=True, cbar=False)
plt.xlabel('predicted value')
plt.ylabel('true value')

We make different mistakes than with `LogisticRegression()`. 

It might be time to go back to the model hyperparameters, or try a gradient boosted tree.

Since we are overfitting, how can we adjust the hyperparameters to reduce model complexity?

In [ ]:
depth = [3, 5, 7, 9]
# Calculate the average cross-validation training and validation scores for each max_depth
# Hint: You should be using a for loop and storing your results in lists
train_score = []
val_score = []

for d in depth:
    model = RandomForestClassifier(random_state=0, max_depth=d)
    scores = cross_validate(model, X_train, y_train, cv=5, scoring='accuracy', return_train_score=True)
    train_score.append(scores['train_score'].mean())
    val_score.append(scores['test_score'].mean())
    
print('Training scores: ', train_score)
print('Validation scores: ', val_score)

**Question**: Which max_depth gives us the best results?

### Summarize the different scores for the best model

In [ ]:
# Create and train the best model
model = RandomForestClassifier(random_state=0, max_depth=3)
model.fit(X_train, y_train)

# Get cross-validation scores
scores = cross_validate(model, X_train, y_train, cv=5, 
                        scoring='accuracy',
                       return_train_score=True)

# Fill in the blanks (...)
print('Training Accuracy (all data): {:.3f}'.format(model.score(X_train, y_train)))

print('Training Accuracy (cross-validation): {:.3f}'.format(scores['train_score'].mean()))

print('Validation Accuracy (cross-validation): {:.3f}'.format(scores['test_score'].mean()))

print('Test Accuracy (new data): {:.3f}'.format(model.score(X_test, y_test)))